# LeetCode 2169 — Count Operations to Obtain Zero bằng Python

**Link đề:** https://leetcode.com/problems/count-operations-to-obtain-zero/
**Độ khó:** Easy · **Chủ đề:** Math, Simulation

```python
class Solution:
    def countOperations(self, num1: int, num2: int) -> int:
        ...
```

## Mục tiêu học
1. Mô phỏng (simulate) đúng luật bằng `while` và nhánh `if/else`, đếm số thao tác.
2. Nhận ra nút thắt: trừ lặp lại cùng một số nhiều lần, ví dụ `(100000, 1)` cần 100.000 lần trừ.
3. Insight: **trừ liên tiếp = chia lấy thương**, `a // b` lần trừ đưa `a` về `a % b` → đây chính là **thuật toán Euclid** tìm ƯCLN (GCD).
4. Nắm `//`, `%`, `divmod`, hoán đổi `a, b = b, a % b`, đệ quy.
5. Phân tích vì sao Euclid chỉ cần `O(log(min(a, b)))` bước.

## 1. Đề bài

Cho hai số nguyên **không âm** `num1`, `num2`. Mỗi **thao tác**:
- nếu `num1 >= num2`: `num1 = num1 - num2`;
- ngược lại: `num2 = num2 - num1`.

Trả về số thao tác cần để **`num1 == 0` hoặc `num2 == 0`**.

| `num1` | `num2` | Kết quả | Diễn biến |
|:-:|:-:|:-:|---|
| 2 | 3 | `3` | (2,3) → (2,1) → (1,1) → (0,1) |
| 10 | 10 | `1` | (10,10) → (0,10) |

**Constraints:** `0 <= num1, num2 <= 10^5`.

Chú ý: nếu ngay từ đầu một số đã bằng `0` thì **không cần thao tác nào** → trả `0`.

In [1]:
a, b_ = 2, 3
ops = 0
print((a, b_))
while a and b_:
    if a >= b_:
        a -= b_
    else:
        b_ -= a
    ops += 1
    print((a, b_))
print("số thao tác:", ops)

(2, 3)
(2, 1)
(1, 1)
(0, 1)
số thao tác: 3


## 2. Nền tảng Python cần biết

### 2.1. `while` với điều kiện "cả hai khác 0"

- `while a != 0 and b != 0:` — lặp khi **cả hai** còn khác 0.
- Viết gọn: `while a and b:` vì số `0` là **falsy** (được coi là `False`), số khác 0 là truthy.

In [2]:
for a, b_ in [(0, 5), (3, 0), (2, 3)]:
    print((a, b_), "| a and b là truthy?", bool(a and b_), "| a != 0 and b != 0:", a != 0 and b_ != 0)

(0, 5) | a and b là truthy? False | a != 0 and b != 0: False
(3, 0) | a and b là truthy? False | a != 0 and b != 0: False
(2, 3) | a and b là truthy? True | a != 0 and b != 0: True


### 2.2. `-=`, `+=` và nhánh `if / else`

`a -= b` là viết tắt `a = a - b`; `ops += 1` tăng bộ đếm.

### 2.3. `//`, `%` và `divmod`

- `a // b`: thương (số lần `b` "vừa" trong `a`).
- `a % b`: phần dư sau khi trừ `b` nhiều nhất có thể.
- `divmod(a, b)` trả cả hai `(a // b, a % b)`.

Liên hệ: trừ `b` khỏi `a` liên tục cho đến khi `a < b` ⟺ trừ đúng `a // b` lần, còn lại `a % b`.

In [3]:
a, b_ = 17, 5
print("17 // 5 =", a // b_, "| 17 % 5 =", a % b_, "| divmod =", divmod(a, b_))
x, count = a, 0
while x >= b_:
    x -= b_
    count += 1
print("trừ liên tiếp:", count, "lần, còn lại", x)

17 // 5 = 3 | 17 % 5 = 2 | divmod = (3, 2)
trừ liên tiếp: 3 lần, còn lại 2


### 2.4. Hoán đổi bằng tuple: `a, b = b, a % b`

Vế phải được tính **trước** (dùng giá trị cũ của `a`, `b`), sau đó mới gán cùng lúc. Không cần biến tạm.

In [4]:
a, b_ = 17, 5
a, b_ = b_, a % b_
print(a, b_)       # 5, 2

5 2


### 2.5. Ước chung lớn nhất (GCD) và `math.gcd`

ƯCLN của `a`, `b` là số lớn nhất chia hết cả hai. Thuật toán **Euclid**: `gcd(a, b) = gcd(b, a % b)`, dừng khi `b = 0`. Phép trừ trong đề là phiên bản "chậm" của Euclid (Euclid bằng phép trừ).

In [5]:
import math
print(math.gcd(12, 18), math.gcd(2, 3), math.gcd(10, 10))

6 1 10


## 3. Cách 1 — Mô phỏng trực tiếp (simulation)

Làm đúng như lời đề, đếm số vòng.

In [6]:
def count_operations_simulate(num1: int, num2: int) -> int:
    ops = 0
    while num1 != 0 and num2 != 0:
        if num1 >= num2:
            num1 -= num2
        else:
            num2 -= num1
        ops += 1
    return ops

print(count_operations_simulate(2, 3))     # 3
print(count_operations_simulate(10, 10))   # 1
print(count_operations_simulate(0, 7))     # 0

3
1
0


### Chi phí và nút thắt

Số vòng có thể rất lớn khi một số **nhỏ hơn hẳn** số kia: `(100000, 1)` cần trừ 1 đúng 100.000 lần. Worst case `O(max(num1, num2))`. Với constraint `10^5` vẫn chạy được, nhưng nếu đề tăng lên `10^9` thì không ổn.

In [7]:
for a, b_ in [(2, 3), (100, 1), (100000, 1), (99999, 100000)]:
    print(f"({a}, {b_}) -> {count_operations_simulate(a, b_)} thao tác")

(2, 3) -> 3 thao tác
(100, 1) -> 100 thao tác
(100000, 1) -> 100000 thao tác
(99999, 100000) -> 100000 thao tác


## 4. Insight: gom các phép trừ giống nhau thành một phép chia

Khi `a >= b`, ta sẽ trừ `b` khỏi `a` **liên tục** cho tới khi `a < b`. Số lần trừ đúng bằng `a // b`, và `a` trở thành `a % b`. Sau đó vai trò hai số đổi chỗ.

Ví dụ `(17, 5)`:
- trừ 5 ba lần: `17 → 12 → 7 → 2` ⇒ `17 // 5 = 3` thao tác, `a = 17 % 5 = 2`.
- giờ `(2, 5)`: trừ 2 hai lần ở `b`: `5 → 3 → 1` ⇒ `5 // 2 = 2` thao tác, `b = 1`.
- `(2, 1)`: trừ 1 hai lần: `2 → 1 → 0` ⇒ `2 // 1 = 2` thao tác. Dừng.
- Tổng `3 + 2 + 2 = 7`.

In [8]:
print("mô phỏng (17, 5):", count_operations_simulate(17, 5))
print("tổng thương:", 17 // 5 + 5 // 2 + 2 // 1)

mô phỏng (17, 5): 7
tổng thương: 7


## 5. Cách 2 — Lời giải chính: Euclid đếm thương

Luôn giữ `a >= b` bằng cách đổi chỗ, rồi:

**Đoạn 1:** `a, b = max(num1, num2), min(num1, num2)`.

**Đoạn 2:** `while b > 0:`

**Đoạn 3:** `ops += a // b` (số lần trừ `b` liên tiếp).

**Đoạn 4:** `a, b = b, a % b` (sau khi trừ, `a % b < b` nên vai trò đổi).

Khi `b == 0` ⇒ một số đã về 0 ⇒ dừng.

In [9]:
def count_operations_euclid(num1: int, num2: int) -> int:
    a, b = max(num1, num2), min(num1, num2)   # Đoạn 1
    ops = 0
    while b > 0:                               # Đoạn 2
        ops += a // b                          # Đoạn 3
        a, b = b, a % b                        # Đoạn 4
    return ops

print(count_operations_euclid(2, 3))       # 3
print(count_operations_euclid(10, 10))     # 1
print(count_operations_euclid(0, 7))       # 0
print(count_operations_euclid(100000, 1))  # 100000 — nhưng chỉ 1 vòng lặp!

3
1
0
100000


Thật ra không cần sắp `max/min` trước: nếu `a < b` thì `a // b = 0` và `a, b = b, a % b = b, a` — tự động đổi chỗ, tốn thêm một vòng vô hại. Nhưng sắp trước giúp code dễ hiểu hơn.

In [10]:
a, b_ = 3, 17
print("a // b =", a // b_, "| (b, a % b) =", (b_, a % b_), "<- tự đổi chỗ")

a // b = 0 | (b, a % b) = (17, 3) <- tự đổi chỗ


## 6. TRACE từng bước

`num1 = 17`, `num2 = 5`:

| Vòng | `a` | `b` | `a // b` (thao tác thêm) | `ops` sau | `a % b` | `(a, b)` mới |
|:-:|:-:|:-:|:-:|:-:|:-:|---|
| 1 | 17 | 5 | 3 | 3 | 2 | (5, 2) |
| 2 | 5 | 2 | 2 | 5 | 1 | (2, 1) |
| 3 | 2 | 1 | 2 | 7 | 0 | (1, 0) |
| — | 1 | 0 | dừng | **7** | | |

So với mô phỏng 7 thao tác — chỉ cần **3 vòng lặp**.

In [11]:
def trace_euclid(num1, num2):
    a, b = max(num1, num2), min(num1, num2)
    ops = 0
    turn = 1
    print(f"bắt đầu: a={a}, b={b}")
    while b > 0:
        q, r = divmod(a, b)
        ops += q
        print(f"Vòng {turn}: a={a:>3}, b={b:>3} -> trừ {q} lần, còn {r}; ops={ops}; (a, b) mới = ({b}, {r})")
        a, b = b, r
        turn += 1
    print("Kết quả:", ops)
    return ops

trace_euclid(17, 5)

bắt đầu: a=17, b=5
Vòng 1: a= 17, b=  5 -> trừ 3 lần, còn 2; ops=3; (a, b) mới = (5, 2)
Vòng 2: a=  5, b=  2 -> trừ 2 lần, còn 1; ops=5; (a, b) mới = (2, 1)
Vòng 3: a=  2, b=  1 -> trừ 2 lần, còn 0; ops=7; (a, b) mới = (1, 0)
Kết quả: 7


7

In [12]:
def trace_simulate(num1, num2, limit=12):
    ops = 0
    while num1 and num2:
        before = (num1, num2)
        if num1 >= num2:
            num1 -= num2
            act = "num1 -= num2"
        else:
            num2 -= num1
            act = "num2 -= num1"
        ops += 1
        if ops <= limit:
            print(f"Thao tác {ops}: {before} --{act}--> {(num1, num2)}")
    print("Tổng:", ops)
    return ops

trace_simulate(17, 5)

Thao tác 1: (17, 5) --num1 -= num2--> (12, 5)
Thao tác 2: (12, 5) --num1 -= num2--> (7, 5)
Thao tác 3: (7, 5) --num1 -= num2--> (2, 5)
Thao tác 4: (2, 5) --num2 -= num1--> (2, 3)
Thao tác 5: (2, 3) --num2 -= num1--> (2, 1)
Thao tác 6: (2, 1) --num1 -= num2--> (1, 1)
Thao tác 7: (1, 1) --num1 -= num2--> (0, 1)
Tổng: 7


7

## 7. Code nộp LeetCode — Cách 2

In [13]:
class Solution:
    def countOperations(self, num1: int, num2: int) -> int:
        ops = 0
        while num1 and num2:
            if num1 < num2:
                num1, num2 = num2, num1
            ops += num1 // num2
            num1 %= num2
        return ops

sol = Solution()
print(sol.countOperations(2, 3), sol.countOperations(10, 10), sol.countOperations(0, 5))

3 1 0


Phiên bản này giữ đúng tên `num1`, `num2` của đề: nếu `num1 < num2` thì đổi chỗ, rồi `num1 %= num2` (viết tắt `num1 = num1 % num2`).

## 8. Cách 3 — Đệ quy (recursion)

Euclid có dạng đệ quy rất tự nhiên:
- **Trường hợp cơ sở (base case):** nếu `b == 0` → `0` thao tác.
- **Bước đệ quy:** `ops(a, b) = a // b + ops(b, a % b)` (với `a >= b`).

In [14]:
def count_operations_recursive(a: int, b: int) -> int:
    if a < b:
        a, b = b, a
    if b == 0:
        return 0
    return a // b + count_operations_recursive(b, a % b)

print(count_operations_recursive(2, 3), count_operations_recursive(17, 5), count_operations_recursive(10, 10))

3 7 1


In [15]:
class Solution:
    def countOperations(self, num1: int, num2: int) -> int:
        if num1 < num2:
            num1, num2 = num2, num1
        if num2 == 0:
            return 0
        return num1 // num2 + self.countOperations(num2, num1 % num2)

print(Solution().countOperations(17, 5))

7


Độ sâu đệ quy chỉ `O(log(min))` (vài chục tầng), không lo vượt giới hạn đệ quy của Python (mặc định khoảng 1000). Nhưng nếu viết đệ quy cho **Cách 1** (mỗi phép trừ một tầng) thì `(100000, 1)` sẽ gây `RecursionError`:

In [16]:
import sys
def simulate_recursive(a, b):
    if a == 0 or b == 0:
        return 0
    if a >= b:
        return 1 + simulate_recursive(a - b, b)
    return 1 + simulate_recursive(a, b - a)

print("giới hạn đệ quy:", sys.getrecursionlimit())
print(simulate_recursive(17, 5))
try:
    simulate_recursive(100000, 1)
except RecursionError as e:
    print("RecursionError:", str(e)[:40], "...")

giới hạn đệ quy: 1000
7
RecursionError: maximum recursion depth exceeded ...


## 9. Vì sao đúng? (invariant)

**Tương đương từng khối:** khi `a >= b > 0`, đề bắt trừ `b` khỏi `a` ở mỗi thao tác **chừng nào `a >= b`**. Sau `q` lần trừ, `a` thành `a - q·b`; điều kiện tiếp tục là `a - q·b >= b`. Lần dừng đầu tiên xảy ra khi `a - q·b < b` ⇒ `q = a // b`, phần còn lại `a % b`. Trong khối này **không** có thao tác nào trừ vào `b` (vì `a >= b` suốt khối, trừ lần cuối thì `a` có thể `< b`).

Lưu ý trường hợp `a == b`: đề trừ vào `num1` (`num1 >= num2`) → `a = 0`; Euclid cũng cho `a // b = 1`, `a % b = 0`. Khớp.

**Invariant Cách 2:** trước mỗi vòng `while`, `ops` = số thao tác đề đã thực hiện để đưa cặp ban đầu về cặp `{a, b}` hiện tại (không quan tâm thứ tự). Mỗi vòng cộng đúng số thao tác của một khối. Vòng dừng khi `b = 0` ⟺ quá trình của đề dừng.

**Dừng:** `b` giảm nghiêm ngặt (`a % b < b`) và không âm ⇒ hữu hạn vòng.

## 10. Độ phức tạp

Gọi `m = min(num1, num2)`, `M = max(num1, num2)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | `O(M / m + ...)`, worst `O(M)` (ví dụ `(M, 1)`) | `O(1)` | dễ nghĩ nhất |
| 2. Euclid lặp | `O(log m)` | `O(1)` | lời giải chính |
| 3. Euclid đệ quy | `O(log m)` | `O(log m)` (stack) | ngắn, đẹp |

**Vì sao Euclid `O(log m)`?** Sau **hai** vòng liên tiếp, số nhỏ hơn giảm ít nhất một nửa: nếu `b <= a/2` thì `a % b < b <= a/2`; nếu `b > a/2` thì `a % b = a - b < a/2`. Vậy số vòng `≤ 2·log2(m) + 1`. Trường hợp tệ nhất là hai số **Fibonacci liên tiếp**.

In [17]:
def euclid_rounds(a, b):
    rounds = 0
    while b:
        a, b = b, a % b
        rounds += 1
    return rounds

fib = [1, 1]
while fib[-1] < 100000:
    fib.append(fib[-1] + fib[-2])
a, b_ = fib[-2], fib[-3]
print(f"Fibonacci liên tiếp ({a}, {b_}): Euclid {euclid_rounds(a, b_)} vòng, mô phỏng {count_operations_simulate(a, b_)} thao tác")
print(f"(100000, 1): Euclid {euclid_rounds(100000, 1)} vòng, mô phỏng {count_operations_simulate(100000, 1)} thao tác")

Fibonacci liên tiếp (75025, 46368): Euclid 23 vòng, mô phỏng 24 thao tác
(100000, 1): Euclid 1 vòng, mô phỏng 100000 thao tác


Thú vị: với hai số Fibonacci liên tiếp, gần như mọi khối chỉ có **1** lần trừ (chỉ khối cuối `(2, 1)` có 2 lần), nên số thao tác của đề chỉ hơn số vòng Euclid đúng 1 — đây là trường hợp mô phỏng gần như không thua Euclid. Ngược lại `(100000, 1)` là trường hợp Euclid thắng đậm nhất.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. Một hoặc cả hai bằng 0.
3. Một số là bội số của số kia (`(12, 4)` → 3).
4. Số lớn + số 1 (worst case mô phỏng).
5. Fibonacci liên tiếp.
6. Đối chiếu mọi cặp nhỏ `0..60` giữa các cách.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

impls = [count_operations_simulate, count_operations_euclid, count_operations_recursive, Solution().countOperations]
cases = [
    ("đề ví dụ 1", 2, 3, 3),
    ("đề ví dụ 2", 10, 10, 1),
    ("num1 = 0", 0, 5, 0),
    ("num2 = 0", 5, 0, 0),
    ("cả hai = 0", 0, 0, 0),
    ("bội số", 12, 4, 3),
    ("số lớn và 1", 100000, 1, 100000),
    ("ví dụ trace", 17, 5, 7),
    ("Fibonacci 89, 55", 89, 55, 10),
]
for name, a, b_, exp in cases:
    for f in impls:
        assert f(a, b_) == exp, (name, f)
    check(name, count_operations_euclid(a, b_), exp)

đề ví dụ 1             PASS (3)
đề ví dụ 2             PASS (1)
num1 = 0               PASS (0)
num2 = 0               PASS (0)
cả hai = 0             PASS (0)
bội số                 PASS (3)
số lớn và 1            PASS (100000)
ví dụ trace            PASS (7)
Fibonacci 89, 55       PASS (10)


In [19]:
checked = 0
for a in range(0, 61):
    for b_ in range(0, 61):
        expected = count_operations_simulate(a, b_)
        for f in impls:
            assert f(a, b_) == expected, (a, b_, f)
        checked += 1
print("Đối chiếu", checked, "cặp (0..60)²: PASS")

Đối chiếu 3721 cặp (0..60)²: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dừng khi **cả hai** bằng 0 (dùng `or` thay vì `and`)

In [20]:
def wrong_or(num1, num2, guard=10**6):
    ops = 0
    while (num1 != 0 or num2 != 0) and ops < guard:   # guard để không treo máy
        if num1 >= num2:
            num1 -= num2
        else:
            num2 -= num1
        ops += 1
    return ops

print("sai :", wrong_or(2, 3), "(chạm guard = lặp vô hạn) | đúng:", count_operations_simulate(2, 3))

sai : 1000000 (chạm guard = lặp vô hạn) | đúng: 3


**Vì sao sai:** khi `num1 = 0`, `num2 = 1`: `0 >= 1` sai → `num2 -= 0` → không đổi gì → **lặp vô hạn**. Đề dừng khi **một trong hai** bằng 0. **Sửa:** `while num1 and num2`.

### Lỗi 2: Đếm số vòng Euclid thay vì tổng thương

In [21]:
def wrong_count_rounds(num1, num2):
    a, b = max(num1, num2), min(num1, num2)
    ops = 0
    while b:
        a, b = b, a % b
        ops += 1            # sai: mỗi vòng có thể gồm nhiều lần trừ
    return ops

print("sai :", wrong_count_rounds(17, 5), "| đúng:", count_operations_euclid(17, 5))

sai : 3 | đúng: 7


**Vì sao sai:** một vòng Euclid gói `a // b` thao tác. **Sửa:** `ops += a // b`.

### Lỗi 3: Chia cho 0 khi một số bằng 0 từ đầu

In [22]:
def wrong_div_zero(num1, num2):
    a, b = max(num1, num2), min(num1, num2)
    ops = a // b                 # chưa kiểm tra b == 0
    return ops

try:
    wrong_div_zero(5, 0)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

ZeroDivisionError: integer division or modulo by zero


**Vì sao sai:** phải kiểm tra điều kiện dừng **trước** khi chia. **Sửa:** đặt phép chia bên trong `while b > 0`.

### Lỗi 4: Cập nhật tuần tự thay vì đồng thời

In [23]:
a, b_ = 17, 5
a = b_          # a = 5
b_ = a % b_     # dùng a MỚI: 5 % 5 = 0  -> sai
print("tuần tự:", (a, b_))
a, b_ = 17, 5
a, b_ = b_, a % b_
print("đồng thời:", (a, b_))

tuần tự: (5, 0)
đồng thời: (5, 2)


**Vì sao sai:** sau dòng đầu, `a` đã mất giá trị cũ. **Sửa:** dùng gán tuple `a, b = b, a % b`, hoặc biến tạm.

## 13. Mẹo nhận dạng pattern

- "**Trừ số nhỏ khỏi số lớn lặp lại**" → đó là **Euclid bằng phép trừ**; thay bằng `//` và `%`.
- Mô phỏng có nhiều bước **giống hệt nhau liên tiếp** → tìm cách **gom khối** bằng phép chia/nhân.
- Điều kiện dừng "một trong hai bằng 0" → `and` trong `while` (lặp khi cả hai khác 0).
- Euclid có độ phức tạp `O(log)`; worst case là cặp Fibonacci.

Bài liên quan: #1979 Find Greatest Common Divisor of Array, #365 Water and Jug Problem, #2001 Number of Pairs of Interchangeable Rectangles, #1342 Number of Steps to Reduce a Number to Zero.

## 14. Biến thể

### 14.1. Trả về cả GCD và số thao tác
Khi quá trình dừng, số **còn lại khác 0** chính là `gcd(num1, num2)` (nếu không cả hai bằng 0 ngay từ đầu).

In [24]:
def ops_and_gcd(num1, num2):
    a, b = max(num1, num2), min(num1, num2)
    ops = 0
    while b:
        ops += a // b
        a, b = b, a % b
    return ops, a

for pair in [(17, 5), (12, 18), (10, 10)]:
    print(pair, "->", ops_and_gcd(*pair), "| math.gcd =", math.gcd(*pair))

(17, 5) -> (7, 1) | math.gcd = 1
(12, 18) -> (3, 6) | math.gcd = 6
(10, 10) -> (1, 10) | math.gcd = 10


### 14.2. Liên hệ: số bước của **phân số liên tục** (continued fraction)
Dãy thương `a // b` của Euclid chính là các hệ số phân số liên tục của `a / b`. Ví dụ `17/5 = 3 + 1/(2 + 1/2)` → thương `[3, 2, 2]`, tổng `7` = đáp án bài!

In [25]:
def continued_fraction(a, b):
    coeffs = []
    while b:
        coeffs.append(a // b)
        a, b = b, a % b
    return coeffs

cf = continued_fraction(17, 5)
print(cf, "tổng =", sum(cf))

[3, 2, 2] tổng = 7


### 14.3. LeetCode 1342 — Number of Steps to Reduce a Number to Zero
Một mô phỏng khác: chẵn thì chia 2, lẻ thì trừ 1. Cũng có thể "gom" bằng bit: số bước = `bit_length - 1 + popcount`.

In [26]:
def number_of_steps(num):
    steps = 0
    while num:
        num = num // 2 if num % 2 == 0 else num - 1
        steps += 1
    return steps

def number_of_steps_bits(num):
    return 0 if num == 0 else num.bit_length() - 1 + bin(num).count("1")

print([number_of_steps(x) for x in (14, 8, 123, 0)], [number_of_steps_bits(x) for x in (14, 8, 123, 0)])

[6, 4, 12, 0] [6, 4, 12, 0]


## 15. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Tính tay
`countOperations(7, 20)` = ? Hãy viết dãy `(a, b)` và các thương.

### Bài 2 — Đoán output
1. `divmod(100, 7)` = ?
2. `count_operations_simulate(0, 0)` = ?
3. `continued_fraction(13, 8)` = ?

### Bài 3 — Điền chỗ trống (Cách 2)

In [27]:
def exercise_count_operations(num1: int, num2: int) -> int:
    ops = 0
    # TODO 1: while cả hai khác 0
    # TODO 2: nếu num1 < num2 thì đổi chỗ
    # TODO 3: ops += num1 // num2; num1 %= num2
    return ops

# assert exercise_count_operations(2, 3) == 3
# assert exercise_count_operations(17, 5) == 7

### Bài 4 — Tự viết hàm `my_gcd(a, b)` bằng Euclid (không dùng `math`)

In [28]:
def exercise_my_gcd(a: int, b: int) -> int:
    # TODO
    pass

# assert exercise_my_gcd(12, 18) == 6
# assert exercise_my_gcd(7, 0) == 7

### Bài 5 — Biến thể: trả về danh sách các cặp `(num1, num2)` sau **mỗi khối** (mỗi vòng Euclid)
Ví dụ `(17, 5)` → `[(2, 5), (2, 1), (0, 1)]` — giữ đúng vai trò `num1`/`num2` của đề (không đổi chỗ).

In [29]:
def exercise_block_states(num1: int, num2: int) -> list[tuple[int, int]]:
    # TODO
    pass

# assert exercise_block_states(17, 5) == [(2, 5), (2, 1), (0, 1)]
# assert exercise_block_states(10, 10) == [(0, 10)]

### Bài 6 — Mini challenge
Trong tất cả cặp `1 <= num1, num2 <= 1000`, cặp nào có **số vòng Euclid** nhiều nhất? (Gợi ý: Fibonacci.) Viết hàm trả về số vòng lớn nhất đó.

In [30]:
def exercise_max_euclid_rounds(limit: int) -> int:
    # TODO: thử mọi cặp hoặc suy luận bằng Fibonacci
    pass

# assert exercise_max_euclid_rounds(1000) == 14

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đổi thành `(20, 7)` | thương lần lượt 2, 1, 6 | `(20,7) → (7,6) → (6,1) → (1,0)` |
| 2 | `100 = 7·14 + 2` | cả hai 0 → không vòng nào | `13/8 = 1 + 5/8`, ... |
| 3 | `while num1 and num2` | đổi chỗ bằng tuple | `%=` |
| 4 | `while b: a, b = b, a % b` | trả `a` | `gcd(a, 0) = a` |
| 5 | Mỗi vòng biết ai `>=` ai | nếu `num1 >= num2`: `num1 %= num2` | còn lại `num2 %= num1`; append `(num1, num2)` |
| 6 | Fibonacci cho số vòng lớn nhất | hai Fibonacci lớn nhất ≤ 1000 là `987, 610` | đếm vòng bằng `euclid_rounds` |

## 16. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `(20,7)`: 20 // 7 = 2, dư 6 → `(7,6)`: 1, dư 1 → `(6,1)`: 6, dư 0. Tổng `2 + 1 + 6 = 9`.

**Bài 2:** `(14, 2)`; `0`; `[1, 1, 1, 1, 2]` (13/8 = 1 + 1/(1 + 1/(1 + 1/(1 + 1/2)))).

In [31]:
trace_euclid(7, 20)
print(divmod(100, 7), count_operations_simulate(0, 0), continued_fraction(13, 8))

bắt đầu: a=20, b=7
Vòng 1: a= 20, b=  7 -> trừ 2 lần, còn 6; ops=2; (a, b) mới = (7, 6)
Vòng 2: a=  7, b=  6 -> trừ 1 lần, còn 1; ops=3; (a, b) mới = (6, 1)
Vòng 3: a=  6, b=  1 -> trừ 6 lần, còn 0; ops=9; (a, b) mới = (1, 0)
Kết quả: 9
(14, 2) 0 [1, 1, 1, 1, 2]


In [32]:
def answer_count_operations(num1, num2):
    ops = 0
    while num1 and num2:
        if num1 < num2:
            num1, num2 = num2, num1
        ops += num1 // num2
        num1 %= num2
    return ops


def answer_my_gcd(a, b):
    while b:
        a, b = b, a % b
    return a


def answer_block_states(num1, num2):
    states = []
    while num1 and num2:
        if num1 >= num2:
            num1 %= num2
        else:
            num2 %= num1
        states.append((num1, num2))
    return states


def answer_max_euclid_rounds(limit):
    best = 0
    for a in range(1, limit + 1):
        for b in range(1, a + 1):
            best = max(best, euclid_rounds(a, b))
    return best


assert answer_count_operations(2, 3) == 3
assert answer_count_operations(17, 5) == 7
assert answer_my_gcd(12, 18) == 6
assert answer_my_gcd(7, 0) == 7
assert answer_block_states(17, 5) == [(2, 5), (2, 1), (0, 1)]
assert answer_block_states(10, 10) == [(0, 10)]
assert answer_max_euclid_rounds(1000) == 14
assert euclid_rounds(987, 610) == 14
for a in range(0, 40):
    for b_ in range(0, 40):
        assert answer_count_operations(a, b_) == count_operations_simulate(a, b_)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


Lưu ý Bài 5: với `a == b`, đề trừ vào `num1` (vì `num1 >= num2`), nên khối là `num1 %= num2` → `num1 = 0`. Nhánh `>=` trong đáp án giữ đúng luật này.

## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countOperations(self, num1: int, num2: int) -> int:
        ops = 0
        while num1 and num2:
            if num1 < num2:
                num1, num2 = num2, num1
            ops += num1 // num2
            num1 %= num2
        return ops
```

**Khung tư duy:** mô phỏng đúng luật → thấy bước lặp giống nhau → gom bằng `//` và `%` → nhận ra Euclid → `O(log)`.

**Checklist trước khi nộp:**
- [ ] Điều kiện dừng: **một trong hai** bằng 0 (`while a and b`)?
- [ ] Input có số 0 ngay từ đầu → trả `0`, không chia cho 0?
- [ ] Cộng `a // b`, không cộng `1` mỗi vòng?
- [ ] Cập nhật đồng thời bằng tuple?
- [ ] Test `(10, 10)`, `(100000, 1)`, Fibonacci.